In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
import os
from IPython.display import display, update_display, Markdown
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from openai import OpenAI
import requests
import torch

In [ ]:
model = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
system_message = """
You will produce synthetic dataset of the topic given by the user, be accurate, give atleast 10 data of the topic and maximum 50 and be creative,
give in markdown format and without code blocks. its like you are giving dataset of that topic to the user which can be used to train some model.
"""

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16

)

In [ ]:
def gen(topic):
 user_prompt = f"""
 I want to get data about the topic of {topic}
 """
 messages = [{"role" : "user", "content" : user_prompt}, {"role" : "system", "content" : system_message}]

 tokenizer = AutoTokenizer.from_pretrained(model)
 tokenizer.pad_token = tokenizer.eos_token
 inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
 streamer = TextStreamer(tokenizer)
 Modeled = AutoModelForCausalLM.from_pretrained(model, device_map = "auto", quantization_config = quant_config)
 outputs = Modeled.generate(inputs, max_new_tokens = 3000)
 texts = tokenizer.decode(outputs[0], skip_special_tokens=True)
 return texts

In [ ]:
import gradio as gr

In [ ]:
gr.Interface(
    fn=gen,
    inputs=gr.Textbox(label="Topic", lines=4),
    outputs=gr.Markdown(label="Dataset")
).launch()